# Rover on Rough Terrain — Planning Challenge

A point rover lives on a 2D arena. It must drive from **start** to **goal**.

You are given:
- `field` — a `(H, W, 3)` terrain map with 3 channels you can see:
  - **ch 0 — rocks**: hard obstacles, binary `0` (no rock) / `1` (rock); lethal, must avoid
  - **ch 1 — mud**: continuous 0..1
  - **ch 2 — slope**: continuous 0..1
- `demos` — a handful of **expert trajectories** (lists of `(x, y)` points).
  Each demo runs between its *own* start/goal pair (NOT necessarily yours), so
  no single demo is the answer — they're there to reveal the terrain cost.
- `start`, `goal` — the 2D points YOUR path must connect (`x = column`, `y = row`).

**Motion model:** the rover is a point in continuous `(x, y)` space; your path
is any polyline of waypoints (not restricted to a grid). The metric samples
along every segment, so sparse waypoints can't skip over a rock or costly cell.

Your job: a path that reaches the goal and is about as terrain-cheap as the
experts'. A pre-provided baseline cost lets you run a planner immediately, but
its weights are wrong — so plan under it and you'll cost too much. The demos
reveal what the experts actually avoid.

**Two TODOs:** (1) implement the planner, (2) learn the cost from the demos.
Partial progress is fine — talk through your choices as you go.

### Setup &mdash; load `field`, `demos`, `start`, `goal`, metric metadata

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# Clone the data repo when running on Colab (no-op if files already exist).
if not os.path.exists('field.npy'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field.npy')                          # (H, W, 3)
demos = np.load('demos.npy', allow_pickle=True)       # object array of (T,2)
meta  = np.load('meta.npz')
start, goal = meta['start'], meta['goal']
H, W = field.shape[:2]
ROCK, MUD, SLOPE = 0, 1, 2
print('field', field.shape, '| demos', len(demos),
      '| start', start, 'goal', goal)

### Visualization &mdash; `show(paths, cost_map, title)`, plus one initial render

In [ ]:
def show(paths=None, cost_map=None, title=''):
    fig, ax = plt.subplots(figsize=(6, 6))
    # Default to the true cost map when no cost_map is passed, so both views
    # use the same viridis colormap.
    grid = np.asarray(cost_map if cost_map is not None else meta['true_cost'], float)
    # Rocks carry a huge sentinel/penalty that would swamp the color scale and
    # flatten all the mud/slope variation. Scale the colormap to the terrain
    # (non-rock) range and render rock cells in the colormap's "over" color.
    rock = field[..., ROCK] == 1
    terrain = grid[~rock]
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_over('0.6')   # rocks shown distinctly (gray)
    vmin = float(terrain.min()) if terrain.size else 0.0
    vmax = float(terrain.max()) if terrain.size else 1.0
    ax.imshow(grid, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
    for d in demos:
        d = np.asarray(d)
        ax.plot(d[:, 0], d[:, 1], '0.7', lw=1, alpha=.7)
    if paths is not None:
        if isinstance(paths, np.ndarray) and paths.ndim == 2:
            paths = [paths]
        for p in paths:
            p = np.asarray(p)
            ax.plot(p[:, 0], p[:, 1], 'lime', lw=2)
    ax.plot(*start, 'wo', ms=8); ax.plot(*goal, 'w*', ms=14)
    ax.set_title(title); ax.set_xlim(0, W); ax.set_ylim(0, H)
    plt.show()

show(title='true cost map + expert demos (gray)')

### Pre-built metric &mdash; `path_cost` and `evaluate` (do not edit)

`path_cost(path, cost_map)` integrates *your* cost map along a path (use it to
score rollouts inside your planner); `evaluate(path, cost_map)` is the official
PASS/FAIL scorer. The true terrain cost is hidden behind them &mdash; don't
peek; recovering it from the demos is the point.

In [ ]:
# ===== PRE-BUILT METRIC (do not edit) -- see docstrings for path_cost/evaluate.
# The true per-step terrain cost is an opaque grid in meta['true_cost']; don't
# peek -- recovering it from the demos is the point of the challenge.
EXPERT_COST = float(meta['expert_cost'])
TOL = float(meta['tol'])
_TRUE_COST = meta['true_cost']                 # (H,W) opaque scoring grid
_ROCK = float(meta['rock_sentinel'])           # cells >= this are rock
_STEP = 0.5                                    # path-integral resampling (cells)
# Penalty applied to rock / forbidden cells. A large finite number (not inf)
# so path_cost stays numerically well-behaved -- every rollout the planner
# scores has a comparable cost even if it crosses an obstacle, which avoids
# argmin stalling on a sea of inf values.
ROCK_PENALTY = 1e6
_HI = ROCK_PENALTY / 2                          # cells >= this count as "rock"
# Two views of the hidden grid: rocks penalized (to detect crossings) and rocks
# zeroed (honest terrain-only cost reported against the bar).
_TRUE_COST_PEN  = np.where(_TRUE_COST >= _ROCK, ROCK_PENALTY, _TRUE_COST)
_TRUE_COST_BARE = np.where(_TRUE_COST >= _ROCK, 0.0,          _TRUE_COST)

def _resample_steps(path):
    '''Resample `path` to ~uniform _STEP spacing; return (rx, ry, in_bounds).'''
    path = np.asarray(path, float)
    seg = np.linalg.norm(np.diff(path, axis=0), axis=1)
    L = float(seg.sum())
    if L < 1e-9:
        return None
    s = np.concatenate([[0.0], np.cumsum(seg)])
    n = max(2, int(L / _STEP) + 1)
    u = np.linspace(0, L, n)
    xs = np.interp(u, s, path[:, 0]); ys = np.interp(u, s, path[:, 1])
    rx = np.rint(xs).astype(np.intp)
    ry = np.rint(ys).astype(np.intp)
    in_bounds = (rx >= 0) & (rx <= W - 1) & (ry >= 0) & (ry <= H - 1)
    return rx, ry, in_bounds

def path_cost(path, cost_map):
    '''Integrate cost_map along path; return total cost as a float (lower is
    better, empty path -> inf). Resamples to ~uniform spacing first, so the
    result is independent of waypoint count. Score rollouts with this inside
    your planner. Mark "do not enter" cells with ROCK_PENALTY (not np.inf) so
    every rollout stays comparable; out-of-bounds points are charged the same.
    '''
    cost_map = np.asarray(cost_map, float)
    if cost_map.shape != (H, W):
        raise ValueError(f'cost_map must have shape {(H, W)}, got {cost_map.shape}')
    rs = _resample_steps(path)
    if rs is None:
        return float('inf')
    rx, ry, in_bounds = rs
    # In-bounds points contribute their cell cost; out-of-bounds points are
    # charged ROCK_PENALTY.
    cells = np.where(in_bounds, cost_map[np.clip(ry, 0, H - 1), np.clip(rx, 0, W - 1)],
                     ROCK_PENALTY)
    return float(cells.sum() * _STEP)

def evaluate(path, cost_map):
    '''Official PASS/FAIL scorer. Returns True iff the path passes.

    Checks (1) endpoints near start/goal, (2) the path is feasible under
    `cost_map` (no untraversable cells along it), (3) stays in bounds,
    (4) doesn't touch a rock under the hidden true cost, and (5) terrain
    cost <= expert bar. Prints a one-line summary. To score paths under your
    own cost map without invoking the bar, use path_cost instead.
    '''
    path = np.asarray(path, float)
    start_ok = bool(np.linalg.norm(path[0]  - start) < 5)
    goal_ok  = bool(np.linalg.norm(path[-1] - goal)  < 5)
    rs = _resample_steps(path)
    # Feasibility flags from the resampled points. A "high" cell is one whose
    # cost is >= _HI (half the rock penalty) -- i.e. a rock / forbidden cell.
    if rs is None:
        model_feasible = rock_hit = out_of_bounds = False
    else:
        rx, ry, in_bounds = rs
        out_of_bounds = bool(not in_bounds.all())
        ryc, rxc = np.clip(ry, 0, H - 1), np.clip(rx, 0, W - 1)
        model_feasible = not bool((in_bounds &
                                   (np.asarray(cost_map, float)[ryc, rxc] >= _HI)).any())
        rock_hit = bool((in_bounds & (_TRUE_COST_PEN[ryc, rxc] >= _HI)).any())
    bare_cost = path_cost(path, _TRUE_COST_BARE)   # honest terrain-only cost
    bar = EXPERT_COST * TOL
    success = bool(start_ok and goal_ok and model_feasible
                   and not rock_hit and not out_of_bounds
                   and bare_cost <= bar)
    print(f"start_ok={start_ok} goal_ok={goal_ok} "
          f"model_feasible={model_feasible} rock_hit={rock_hit} "
          f"out_of_bounds={out_of_bounds} "
          f"cost={bare_cost:.1f} (bar={bar:.1f}) -> {'PASS' if success else 'FAIL'}")
    return success

### Pre-provided baseline cost &mdash; `cost_map_baseline` (do not edit)

A runnable linear cost so you can plan immediately. Its weights are
deliberately wrong &mdash; fixing that is TODO 2.

In [ ]:
# Pre-provided baseline cost (do not edit).
def cost_map_baseline(field):
    '''Wrong-but-runnable (H,W) linear cost: 1 + 5*slope (ignores mud),
    rocks at ROCK_PENALTY. Lets you run the planner before learning the cost.'''
    cm = 1.0 + 5.0 * field[..., SLOPE]
    cm[field[..., ROCK] == 1] = ROCK_PENALTY
    return cm

## TODO 1 &mdash; implement the planner (`plan`); details in the cell

In [ ]:
# ---- TODO 1: a sampling-based rollout planner (CPU/numpy first) -----------
# Plan a cheap start->goal path under a given (H,W) cost_map. Suggested scheme
# (you may do otherwise): hold a current T-waypoint trajectory; each iteration
# sample K noisy rollouts around it, score each with path_cost(p, cost_map),
# and step toward the cheap ones (softmax / take-best).
def plan(cost_map, K=256, T=60, iters=30, seed=0):
    '''Return an (T,2) path of (x,y) points from start to goal.'''
    rng = np.random.default_rng(seed)
    # TODO: replace this straight line (which ignores cost_map) with the
    # sampling rollout loop.
    return np.linspace(start, goal, T)

### Run the planner under the baseline cost &mdash; expect `FAIL` (wrong weights)

In [ ]:
cm_base = cost_map_baseline(field)
path = plan(cm_base)
show(path, cost_map=cm_base, title='baseline cost: planned path')
evaluate(path, cm_base)

## TODO 2 &mdash; learn the cost from demos (`cost_map_learned`); details in the cell

In [ ]:
# ---- TODO 2: learn the cost from the expert demos ------------------------
# The baseline charges for the wrong terrain. Use the demos to build a cost map
# that explains the experts' detours, then re-plan. Pick an approach and justify
# it: visitation (cells experts use are cheap), feature regression / IRL (fit
# weights over [mud, slope]), nearest-demo / warping, ...
def cost_map_learned(field, demos):
    '''Return an (H,W) cost map inferred from demos. Keep rocks impassable.'''
    # TODO: use the demos. (Falls back to the wrong baseline for now.)
    return cost_map_baseline(field)

### Run the learned solution &mdash; goal: `PASS`

In [ ]:
cm = cost_map_learned(field, demos)
path = plan(cm)
show(path, cost_map=cm, title='learned: planned path over learned cost map')
show(path, title='learned: planned path over terrain')
evaluate(path, cm)

### Bonus / discussion — make it fast

Your planner rolls out `K` trajectories every iteration. On a real robot
we replan in a closed loop at, say, 50 Hz with `K` in the thousands.

- **Vectorize** the rollout: evaluate all `K` candidates without a Python loop
  (batched numpy, or move the cost lookup + scoring to `torch` on GPU).
- Be ready to discuss: memory layout of the `K × T × 2` buffer, how `K` trades
  off against horizon `T`, host↔device transfer / sync cost, and what your
  per-replan time budget buys you at 50 Hz.

In [ ]:
# ---- BONUS TODO: vectorized / GPU rollout --------------------------------
# Re-implement the per-iteration rollout scoring without a Python loop over K.
# numpy: index cost_map with a (K, T) array of flattened cell ids.
# torch: keep cost_map + samples on cuda; one gather + sum.
# (Optional — sketch it even if you don't finish.)